<a href="https://colab.research.google.com/github/DebStar17/GNSS-Anti-spoofing/blob/main/Copy_of_GNSS_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

# 1. CONFIGURATION
TRAIN_FILE = 'train.csv'
TEST_FILE = 'test.csv'
OUTPUT_FILE = 'final_hybrid_submission.csv'

# Features that focus on relative change, not absolute time
FEATURES = ['CN0', 'Phase_Divergence', 'Doppler_Delta', 'Corr_Symmetry', 'CN0_Stability', 'Tracking_Error', 'Carrier_Doppler_hz']

In [13]:
# 2. DATA ENGINEERING
def load_and_engineer(file_path, is_train=True):
    print(f"--> Loading {file_path}...")
    df = pd.read_csv(file_path, low_memory=False)

    # Garbage Cleanup
    df['Carrier_Doppler_hz'] = pd.to_numeric(df['Carrier_Doppler_hz'], errors='coerce')
    df = df.dropna(subset=['Carrier_Doppler_hz']).copy()

    cols_to_float = ['Pseudorange_m', 'Carrier_phase', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'CN0', 'time']
    for col in cols_to_float:
        df[col] = df[col].astype(float)

    if is_train:
        df['spoofed'] = df['spoofed'].astype(int)

    df = df.sort_values(by=['channel', 'time']).reset_index(drop=True)

    # --- NON-TIME BIASED FEATURES ---

    # 2. Blackout Indicators
    df['Is_Signal_Dead'] = (df['CN0'] == 0).astype(int)
    # df['Blackout_Intensity'] = df.groupby('time')['Is_Signal_Dead'].transform('sum')

    # 3. Physics Relationships
    df['Phase_Divergence'] = df['Pseudorange_m'] - (0.19 * df['Carrier_phase'])
    df['Doppler_Delta'] = df.groupby('channel')['Carrier_Doppler_hz'].diff().fillna(0)


    epsilon = 1e-9
    df['Corr_Symmetry'] = (df['EC'] - df['LC']) / (df['EC'] + df['LC'] + epsilon)
    df['CN0_Stability'] = df.groupby('channel')['CN0'].transform(
        lambda x: x.rolling(window=10, min_periods=1).std().fillna(0)
    )
    df['Tracking_Error'] = np.arctan2(df['PQP'], df['PIP'])

    # # 1. Define 'Healthy' as CN0 > 30. Check if any health existed in the last 60 seconds.
    # # This will be 0 for cold starts and 1 for the spoofed window.
    # df['Was_Recently_Healthy'] = df.groupby('channel')['CN0'].transform(
    #     lambda x: (x.rolling(window=60, min_periods=1).max() > 30).astype(int)
    # )

    # # 2. Identify 'Sudden Death': Current signal is 0, but 5 seconds ago it was > 30.
    # df['CN0_Lag_5'] = df.groupby('channel')['CN0'].shift(5).fillna(0)
    # df['Sudden_Death_Score'] = ((df['CN0'] == 0) & (df['CN0_Lag_5'] > 40)).astype(int)
    # -----------------------

    df = df.fillna(0)
    return df

In [14]:
# 3. PHASE 1: TRAINING
print("\n--- PHASE 1: TRAINING ---")
train_df = load_and_engineer(TRAIN_FILE, is_train=True)

# Sort by time before saving
train_df = train_df.sort_values(by='time')

# Save to intermediate.csv
train_df[['time', 'channel']+ FEATURES].to_csv('intermediate.csv', index=False)
print(f"Intermediate features saved to 'intermediate.csv'")

X_train = train_df[FEATURES]
y_train = train_df['spoofed']

rf_1 = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1) # class_weight='balanced'
train_filter = train_df['CN0'] > 0
rf_1.fit(X_train[train_filter], y_train[train_filter])

# --- ADD THIS TO GENERATE THE LEADERBOARD ---
# 1. Extract importances
importances = rf_1.feature_importances_
feature_names = X_train.columns

# 2. Create a sorted DataFrame
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 3. Print the Leaderboard
print("\nFeature Importance Leaderboard:")
print(feature_importance_df.to_string(index=False))


--- PHASE 1: TRAINING ---
--> Loading train.csv...
Intermediate features saved to 'intermediate.csv'

Feature Importance Leaderboard:
           Feature  Importance
  Phase_Divergence    0.556688
Carrier_Doppler_hz    0.305221
               CN0    0.121168
     Doppler_Delta    0.008152
     CN0_Stability    0.005279
     Corr_Symmetry    0.002141
    Tracking_Error    0.001351


In [15]:
print("\n--- PHASE 2: SCORING & STATE LATCHING ---")
# Load and engineer test data
test_df = load_and_engineer(TEST_FILE, is_train=False)

test_df.to_csv('intermediate_test.csv', index=False)
print(f"Intermediate features saved to 'intermediate_test.csv'")

# Get initial RF confidence using rf_1 (from training cell)
test_df['RF_Prob'] = rf_1.predict_proba(test_df[FEATURES])[:, 1]

# Aggregating: Max for RF probability, Mean for Signal Dead status
result_agg = test_df.groupby('time').agg({
    'RF_Prob': 'max',
    'Is_Signal_Dead': 'mean'
}).reset_index()

spoofed_final = []
confidence_scores = [] # Initialize list for confidence scores
current_state = 0

for i, row in result_agg.iterrows():
    rf_prob = row['RF_Prob']
    is_signal_dead = row['Is_Signal_Dead']

    # --- Determine the 'Spoofed' state ---
    next_state = current_state # Default: maintain current state

    # Using a threshold for mean signal death (e.g., > 0.5) or logic as requested
    if is_signal_dead < 1.0:
      if rf_prob > 0.65:
          next_state = 1
      else:
          next_state = 0

    spoofed_final.append(next_state)

    # --- Calculate Confidence based on final 'Spoofed' state ---
    if is_signal_dead < 1.0:
      if next_state == 1:
          confidence = rf_prob
      else:
          confidence = 1 - rf_prob
    else:
      confidence = 0.5

    confidence_scores.append(confidence)

    current_state = next_state # Update current_state for the next iteration

result_agg['spoofed'] = spoofed_final
result_agg['confidence'] = confidence_scores # Add confidence to the DataFrame

# Final formatting
submission_final = result_agg[['time', 'spoofed', 'confidence']]
submission_final['time'] = submission_final['time'].astype(int)

submission_final.to_csv(OUTPUT_FILE, index=False)
print(f"\nDONE! '{OUTPUT_FILE}' is ready.")


--- PHASE 2: SCORING & STATE LATCHING ---
--> Loading test.csv...
Intermediate features saved to 'intermediate_test.csv'

DONE! 'final_hybrid_submission.csv' is ready.
